# PlantCLEF 2015 Test Leaf Archive

Downloads the official PlantCLEF 2015 annotated test package, extracts leaf-only metadata/images, and stores a compact archive on Google Drive for evaluation notebooks.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. Clone Or Update Project

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


## 3. Download Official Test Package

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

RAW_DIR=/content/plantclef2015_test_raw
DRIVE_RAW=/content/drive/MyDrive/PlantCLEF2015TestDataWithAnnotations.tar.gz
URL=https://lab.plantnet.org/LifeCLEF/PlantCLEF2015/TestPackage/PlantCLEF2015TestDataWithAnnotations.tar.gz

mkdir -p /content/drive/MyDrive
wget -c --tries=20 --timeout=120 --read-timeout=120 "$URL" -O "$DRIVE_RAW"
rm -rf "$RAW_DIR"
mkdir -p "$RAW_DIR"
tar -xzf "$DRIVE_RAW" -C "$RAW_DIR"
find "$RAW_DIR" -type f | wc -l
find "$RAW_DIR" -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' \) | wc -l
find "$RAW_DIR" -type f -iname '*.xml' | wc -l


## 4. Build Leaf-Only Test Bundle

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

RAW_DIR=/content/plantclef2015_test_raw
WORK_DIR=/content/plantclef2015_test_leaf_bundle
RAW_METADATA=/content/plantclef2015_test_metadata_raw.csv
DRIVE_BUNDLE=/content/drive/MyDrive/PlantCLEF2015_leaf_test.tar.gz

rm -rf "$WORK_DIR"
mkdir -p "$WORK_DIR"

python -u -m plant_classifier.data.plantclef_cli \
  --source-root "$RAW_DIR" \
  --image-root "$RAW_DIR" \
  --relative-to "$RAW_DIR" \
  --output "$RAW_METADATA" \
  --content Leaf

python - <<'PY_COPY_LEAF'
import csv
import shutil
from collections import Counter
from pathlib import Path

raw_root = Path('/content/plantclef2015_test_raw')
raw_metadata = Path('/content/plantclef2015_test_metadata_raw.csv')
bundle = Path('/content/plantclef2015_test_leaf_bundle')
image_root = bundle / 'leaf'
metadata_out = bundle / 'metadata.csv'

with raw_metadata.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))

metadata_out.parent.mkdir(parents=True, exist_ok=True)
image_root.mkdir(parents=True, exist_ok=True)

fieldnames = ['image_path', 'family', 'genus', 'species', 'content', 'split', 'source_xml']
with metadata_out.open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for row in rows:
        rel_image = Path(row['image_path'])
        src = raw_root / rel_image
        dst_rel = Path('leaf') / rel_image
        dst = bundle / dst_rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        writer.writerow({
            'image_path': str(dst_rel),
            'family': row.get('family', ''),
            'genus': row.get('genus', ''),
            'species': row.get('species', ''),
            'content': row.get('content', ''),
            'split': row.get('split', 'test') or 'test',
            'source_xml': row.get('source_xml', ''),
        })

print(f'leaf rows: {len(rows)}')
print('genera:', len({row['genus'] for row in rows}))
print('species:', len({row['species'] for row in rows}))
print('split counts:', Counter(row.get('split', 'test') or 'test' for row in rows))
PY_COPY_LEAF

tar -czf "$DRIVE_BUNDLE" -C "$WORK_DIR" .
ls -lh "$DRIVE_BUNDLE"
tar -tzf "$DRIVE_BUNDLE" | sed -n '1,20p'


## 5. Smoke-Extract Bundle

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1

TMP=/content/plantclef2015_test_leaf_check
ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leaf_test.tar.gz
rm -rf "$TMP"
mkdir -p "$TMP"
tar -xzf "$ARCHIVE" -C "$TMP"
test -f "$TMP/metadata.csv"
python - <<'PY_CHECK_BUNDLE'
import csv
from collections import Counter
from pathlib import Path
root = Path('/content/plantclef2015_test_leaf_check')
with (root / 'metadata.csv').open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
missing = [row['image_path'] for row in rows if not (root / row['image_path']).exists()]
print('rows:', len(rows))
print('missing images:', len(missing))
print('content:', Counter(row.get('content', '') for row in rows))
print('split:', Counter(row.get('split', '') for row in rows))
if missing:
    raise FileNotFoundError(missing[:5])
PY_CHECK_BUNDLE
